# LightGBM Sybil Detector — LayerZero

LightGBM, 3-seed ensemble (seeds 42, 123, 456), 5,000-round ceiling with early stopping (50 rounds) on
validation log-loss, on the stratified group split. Fixed settings (`sp.lgbm_model`): `subsample_freq=1`
(required for bagging), `colsample_bytree=0.8`, and `force_col_wise` / `deterministic` so the trees are the
same on every run. `num_leaves`, `learning_rate`, `subsample` and `reg_lambda` are tuned.

Training, evaluation and helper code live in `sybil_pipeline.py` (`sp.fit_lgbm`, `sp.evaluate`), the same
code `11_split_comparison` runs on the original random split.

**Results**: test metrics in section 3, saved to `results/04_lightgbm_sybil.json`.

## Setup

In [ ]:
import warnings
import sybil_pipeline as sp
warnings.filterwarnings('ignore')

DATA_DIR = './data'
FEATS = sp.FEATS   # 62 model features (defined in sybil_pipeline.py)

## 1. Data and split

Stratified group split on gas provision trees (49 % train, 21 % validation, 30 % test); the Sybil class is
upsampled to 1:1 in the training partition only.

In [ ]:
# ── Feature table (steps 1-7; see 00_data_pipeline) and the stratified group split ──
df, _, _ = sp.build_master_df(DATA_DIR)
S = sp.make_splits(df, FEATS, method='group', seed=sp.SEED)   # asserts no gas provision tree spans two partitions
print(sp.split_summary(df, S).to_string(index=False))
print(f"Train after upsampling: {len(S['X_train']):,} rows ({S['y_train'].mean()*100:.1f}% Sybil)\n")
leak = sp.leakage_report(df, S)

## 2. Train (3 seeds, early stopping on validation)

Hyperparameters selected on validation by `02_hyperparameter_search`.

In [ ]:
XGB_SELECTED, LGBM_SELECTED = sp.load_selected_params()
PARAMS = LGBM_SELECTED
print('Tuned hyperparameters:', PARAMS)
m = sp.fit_lgbm(S, PARAMS)

## 3. Evaluate on the test set

Threshold: the F1-maximizing threshold on validation.

In [ ]:
res = sp.evaluate('LightGBM (tuned, 3-seed)', S['y_val'], m['val'], S['y_test'], m['test'])
sp.print_metrics(res)
sp.plot_curves([res])

## 4. Feature importances

Total gain: the reduction in training loss summed over every split on the feature, normalized to sum to 1
for each model and averaged over the three seeds (`sp.gain_importance`). Gain credits correlated features
unevenly; `09_shap_importance` reports SHAP values and sums them by feature family.

In [ ]:
imp_df = sp.gain_importance(m['models'], FEATS)
print('Top 20 features (normalized total gain, mean over 3 seeds):')
print(imp_df.head(20).to_string(index=False))
sp.plot_importance(imp_df, 'LightGBM — Top 20 Feature Importances', '#2E7D32')

## 5. Operating points

Precision, recall and counts on the test set at fixed thresholds and at the threshold chosen on validation
(whose row equals the reported test metrics).

In [ ]:
print(sp.operating_points(S['y_test'], m['test'], res['threshold']).round(4).to_string(index=False))

## Save results

Scalar metrics go to `results/04_lightgbm_sybil.json` with the code commit, so every number in the paper can be traced
to one run. Validation and test probabilities go to `output/` for `06_cross_ensemble_sybil` and `10_tie_out`.

In [ ]:
sp.save_results([res], '04_lightgbm_sybil', leakage=leak,
                extra=dict(params=PARAMS, rounds=m['rounds'],
                           importance=imp_df.set_index('Feature')['NormGain'].to_dict()))
sp.save_predictions(df, S, '04_lightgbm_sybil', prob_lgbm=m)